# Eight-design UiO-66 sampling pilot v2
Prepared, not GPU-executed. Attach UIO66_Sampling_Pilot_v2.zip as a private Dataset;
enable Internet and two T4s. Read bundled docs/UIO66_SAMPLING_PILOT_V2.md first.
512 guest starts plus stripped-empty/reference work. Estimate main 1.1–3.3h on two T4s;
reserve ~1.4–4.2h with representative force comparison. Review allocation and chemistry.
Leave RUN_GPU_PILOT=False until approval. No Run All on an existing busy research notebook.
New working/output paths only. HF_TOKEN via Kaggle Secrets; never print it.


In [ ]:
import hashlib, json, os, subprocess, sys, zipfile
from pathlib import Path
RUN_GPU_PILOT = False  # Set True only after allocation/protocol review.
RUN_FORCE_CHECK = False  # Additional 16 guests, empty/bare/gas refinements.
archives=list(Path('/kaggle/input').rglob('UIO66_Sampling_Pilot_v2.zip'))
assert len(archives)==1, 'Attach exactly one pilot bundle Dataset'
REPO=Path('/kaggle/working/MOF_sampling_v2')
assert not REPO.exists(), 'Choose new extraction directory; preserve prior work'
with zipfile.ZipFile(archives[0]) as z:
    for name in z.namelist():
        target=(REPO/name).resolve()
        assert target.is_relative_to(REPO.resolve()), 'Unsafe archive path'
    z.extractall(REPO)
manifest=json.loads((REPO/'bundle_manifest.json').read_text())
for name,digest in manifest['sha256'].items():
    assert hashlib.sha256((REPO/name).read_bytes()).hexdigest()==digest, name
os.chdir(REPO)
ENV=dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
PLAN=REPO/'data/design/uio66_sampling_pilot_v2.json'
plan=json.loads(PLAN.read_text())
print('Verified bundle; patterns:',[r['pattern'] for r in plan['configurations']])
print('Plan SHA256:',hashlib.sha256(PLAN.read_bytes()).hexdigest())


In [ ]:
%pip install -q "fairchem-core==2.23.0" "ase==3.26.0"


In [ ]:
subprocess.run([sys.executable,'-m','scripts.run_uio66_sampling_pilot',
 '--plan',str(PLAN),'--output','/kaggle/working/uio66_sampling_v2_dry/report.json'],env=ENV,check=True)
print((REPO/'docs/UIO66_SAMPLING_PILOT_V2.md').read_text()[:1600])


## Allocation gate
CPU checks above do not start UMA. Below requires explicit flag after approval.
Archive historical v1 first; use this new notebook/output namespace.
Do not run 64-design expansion or DFT. Stop if template/site assumptions fail review.


In [ ]:
assert RUN_GPU_PILOT, 'Allocation not approved: keep RUN_GPU_PILOT=False'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(2)])
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Cost estimate assumes two T4s'
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
CHECKPOINT=Path(hf_hub_download('facebook/UMA','checkpoints/uma-s-1p2p1.pt',
 revision=plan['checkpoint_revision'],token=os.environ['HF_TOKEN']))
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()==plan['checkpoint_sha256']


In [ ]:
import shutil, time
OUT=Path('/kaggle/working/uio66_sampling_v2')
assert not OUT.exists(), 'Use entirely new output root'
OUT.mkdir()
chunks=[plan['configurations'][i:i+2] for i in range(0,8,2)]
workers=[]
for i,chunk in enumerate(chunks):
    report=OUT/f'worker{i}'/'adsorption.json'
    log=(OUT/f'worker{i}.execution.log').open('w')
    command=[sys.executable,'-m','scripts.run_uio66_sampling_pilot','--execute',
     '--plan',str(PLAN),'--checkpoint',str(CHECKPOINT),'--ids',*[r['id'] for r in chunk],
     '--output',str(report)]
    proc=subprocess.Popen(command,env=dict(ENV,CUDA_VISIBLE_DEVICES=str(i%2)),stdout=log,stderr=subprocess.STDOUT)
    workers.append((proc,log,report))
try:
    while any(p.poll() is None for p,_,_ in workers):
        progress=[]
        for i,(proc,_,report) in enumerate(workers):
            data=json.loads(report.read_text()) if report.exists() else {}
            active=data.get('active_candidate') or {}
            progress.append({'worker':i,'finished':len(data.get('results',[])),
             'active':active.get('id'),'starts_done':len(active.get('starts',[])),'exit':proc.poll()})
        print(progress,flush=True)
        time.sleep(30)
except KeyboardInterrupt:
    for p,_,_ in workers:
        if p.poll() is None: p.terminate()
    for p,_,_ in workers: p.wait()
    raise
finally:
    for _,log,_ in workers: log.close()
    shutil.make_archive('/kaggle/working/uio66_sampling_v2_raw','zip',OUT)
codes=[p.returncode for p,_,_ in workers]
print('Worker exit codes:',codes)
print('Raw/partial archive: /kaggle/working/uio66_sampling_v2_raw.zip')
assert codes==[0]*4, 'Failure retained; inspect/download raw archive before assessment'


In [ ]:
REPORTS=[str(OUT/f'worker{i}'/'adsorption.json') for i in range(4)]
subprocess.run([sys.executable,'-m','scripts.assess_uio66_sampling_pilot','--plan',str(PLAN),
 '--reports',*REPORTS,'--output',str(OUT/'assessment.json')],env=ENV,check=True)
assessment=json.loads((OUT/'assessment.json').read_text())
print(assessment['stability'])
shutil.make_archive('/kaggle/working/uio66_sampling_v2_audited','zip',OUT)


## Representative force study, separately approved
Four registered configurations, targeted/random winners per gas. Additional 16 guest
continuations at 0.02 eV/Å; up to 600 steps. Not a full tight-force resampling study.
Default remains off. Download raw/audited main archive before launching.


In [ ]:
assert RUN_FORCE_CHECK, 'Representative force allocation remains unapproved'
FORCE=Path('/kaggle/working/uio66_sampling_v2_force02')
assert not FORCE.exists()
with Path('/kaggle/working/uio66_sampling_v2_force02.execution.log').open('w') as log:
    proc=subprocess.run([sys.executable,'-m','scripts.run_uio66_sampling_pilot','--execute',
     '--plan',str(PLAN),'--checkpoint',str(CHECKPOINT),'--refine-reports',*REPORTS,
     '--output',str(FORCE/'force_comparison.json')],env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),stdout=log,stderr=subprocess.STDOUT)
if FORCE.exists():
    shutil.make_archive('/kaggle/working/uio66_sampling_v2_force02_raw','zip',FORCE)
assert proc.returncode==0, 'Download partial force archive/log; no success claim'
subprocess.run([sys.executable,'-m','scripts.assess_uio66_sampling_pilot','--plan',str(PLAN),
 '--reports',*REPORTS,'--force-report',str(FORCE/'force_comparison.json'),
 '--output',str(FORCE/'combined_assessment.json')],env=ENV,check=True)
shutil.make_archive('/kaggle/working/uio66_sampling_v2_force02_audited','zip',FORCE)
print(json.loads((FORCE/'combined_assessment.json').read_text())['force_check'])


## Download and review
Download main raw/audited ZIPs, force ZIPs/log and this executed notebook via File →
Download Notebook. Verify saved source/checkpoint/input/plan hashes locally.
Draft-save conflicts do not invalidate local archives but are not successful Save Version.
No automatic full-study go decision. Read the separate chemistry-review/DFT draft;
sampling and force gates require review. Preserve the 51/13 historical split;
all previously scored designs are development evidence, not untouched final tests.
